# ALQAC 2026 — Drive-first Colab RAG runner

This notebook is a thin orchestration layer over the repository CLI. Configure `GITHUB_TOKEN` before Run All; `HF_TOKEN` is optional, and `ALQAC_TEAM_TOKEN` is read only for Private live execution. A new `RUN_ID` pins the current `TuanAnh` commit during `runtime_check`; smoke, full, and resume must use that exact commit. Public runs remain cache-only and nothing is uploaded automatically.

In [ ]:
# Run parameters — edit only this cell between Public and Private runs.
TRACK = 'public'                 # public | private
RUN_MODE = 'runtime_check'      # runtime_check | smoke | full | resume
EXPERIMENT = 'candidate'        # baseline | candidate
EXECUTION_MODE = 'cache-only'   # cache-only for Public; live for Private
RUN_ID = 'public-candidate-v1'
SOURCE_MODE = 'git'             # Unified source workflow; do not change.
GIT_REPO_URL = 'https://github.com/NGBao1608/DL_K23_ALQAC2026.git'
GIT_REF = 'TuanAnh'             # Resolve the initial source pin from this branch.
PUBLIC_SELECTION_RUN_ID = 'public-candidate-v1'
ADAPTER_PATH = None             # Optional PEFT adapter directory on Drive.
APPROVED_MAX_NETWORK_CALLS = None  # Required explicitly for resume.
RETRY_RESERVE = 4
MODEL_CACHE_FREE_GB = 25
PRIVATE_INPUT_SHA256 = '9db83cf98ade7d19df52c60145830bebcc192e064ec830bcd285cefbfddf0252'

assert TRACK in {'public', 'private'}
assert RUN_MODE in {'runtime_check', 'smoke', 'full', 'resume'}
assert EXPERIMENT in {'baseline', 'candidate'}
assert EXECUTION_MODE in {'cache-only', 'live'}
assert SOURCE_MODE == 'git' and GIT_REF == 'TuanAnh'
if TRACK == 'public':
    assert EXECUTION_MODE == 'cache-only', 'Public must not create new API calls.'
if TRACK == 'private':
    assert EXECUTION_MODE == 'live', 'Private production uses live retrieval.'

In [ ]:
import json
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/ALQAC2026')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
for relative in ('source', 'inputs/private', 'cache', 'indexes/law', 'runs/public', 'runs/private', 'exports'):
    (DRIVE_ROOT / relative).mkdir(parents=True, exist_ok=True)

def secret_or_none(name):
    try:
        value = userdata.get(name)
    except Exception:
        return None
    return value.strip() if value and value.strip() else None

required_secrets = ('GITHUB_TOKEN',)
missing_secrets = []
for secret_name in required_secrets:
    secret_value = secret_or_none(secret_name)
    if not secret_value:
        missing_secrets.append(secret_name)
    secret_value = None
if missing_secrets:
    raise ValueError(f'Missing or inaccessible Colab Secrets: {missing_secrets}')
print({'drive_root': str(DRIVE_ROOT), 'track': TRACK, 'run_id': RUN_ID, 'required_secrets_configured': list(required_secrets), 'optional_secrets': ['HF_TOKEN'], 'private_live_secret': 'ALQAC_TEAM_TOKEN'})

## Source bootstrap

A new `RUN_ID` resolves `TuanAnh` once and atomically stores `source_pin.json` on Drive. Subsequent runtime-check, smoke, full, and resume executions clone the branch but detach to that exact commit before installing or running code. Use a new `RUN_ID` to adopt newer source.

In [ ]:
import os
import re
import shutil
import stat
import subprocess

PROJECT_ROOT = Path('/content/alqac2026')
if PROJECT_ROOT.exists():
    assert PROJECT_ROOT.resolve() == Path('/content/alqac2026')
    shutil.rmtree(PROJECT_ROOT)

SOURCE_PIN_PATH = DRIVE_ROOT / 'runs' / TRACK / RUN_ID / 'source_pin.json'
pinned_commit = None
if SOURCE_PIN_PATH.is_file():
    source_pin = json.loads(SOURCE_PIN_PATH.read_text(encoding='utf-8'))
    if source_pin.get('repository') != GIT_REPO_URL or source_pin.get('branch') != GIT_REF:
        raise ValueError(f'Source pin does not match repository/branch: {SOURCE_PIN_PATH}')
    pinned_commit = source_pin.get('commit')
    if not isinstance(pinned_commit, str) or not re.fullmatch(r'[0-9a-fA-F]{40}', pinned_commit):
        raise ValueError(f'Invalid pinned commit: {SOURCE_PIN_PATH}')
    if RUN_MODE != 'runtime_check' and source_pin.get('runtime_check_status') != 'PASS':
        raise ValueError('Pinned source has not passed runtime_check; rerun runtime_check with this RUN_ID.')
    if RUN_MODE in {'full', 'resume'}:
        smoke_manifest_path = DRIVE_ROOT / 'runs' / TRACK / f'{RUN_ID}-smoke' / 'manifest.json'
        if not smoke_manifest_path.is_file():
            raise FileNotFoundError(f'Full/resume requires a completed smoke run: {smoke_manifest_path}')
        smoke_manifest = json.loads(smoke_manifest_path.read_text(encoding='utf-8'))
        if smoke_manifest.get('run', {}).get('status') != 'completed' or smoke_manifest.get('run', {}).get('completed') != 2 or smoke_manifest.get('git_commit') != pinned_commit:
            raise ValueError('Smoke artifact did not complete two cases with the pinned source commit.')
elif RUN_MODE != 'runtime_check':
    raise FileNotFoundError('Run runtime_check first with this RUN_ID to pin a source commit.')

github_token = secret_or_none('GITHUB_TOKEN')
askpass = Path('/content/alqac-git-askpass.sh')
askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$GITHUB_TOKEN" ;; esac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
git_env = {**os.environ, 'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token}
try:
    subprocess.check_call(
        ['git', 'clone', '--branch', GIT_REF, '--single-branch', GIT_REPO_URL, str(PROJECT_ROOT)],
        env=git_env,
    )
finally:
    askpass.unlink(missing_ok=True)
    github_token = None
    git_env = None
if pinned_commit:
    subprocess.check_call(['git', 'checkout', '--detach', pinned_commit], cwd=PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
resolved_commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, text=True, capture_output=True).stdout.strip() or None
resolved_branch = subprocess.check_output(['git', 'branch', '--show-current'], cwd=PROJECT_ROOT, text=True).strip()
if not re.fullmatch(r'[0-9a-fA-F]{40}', resolved_commit or ''):
    raise RuntimeError(f'Unexpected Git commit: {resolved_commit}')
if pinned_commit and resolved_commit != pinned_commit:
    raise RuntimeError(f'Pinned source mismatch: expected={pinned_commit}, actual={resolved_commit}')
if pinned_commit is None:
    if resolved_branch != GIT_REF:
        raise RuntimeError(f'Unexpected Git branch: {resolved_branch}')
    source_pin = {'schema_version': 'source-pin-v1', 'repository': GIT_REPO_URL, 'branch': GIT_REF, 'commit': resolved_commit, 'runtime_check_status': 'pending'}
    SOURCE_PIN_PATH.parent.mkdir(parents=True, exist_ok=True)
    source_pin_temp = SOURCE_PIN_PATH.with_suffix('.json.tmp')
    source_pin_temp.write_text(json.dumps(source_pin, indent=2), encoding='utf-8')
    source_pin_temp.replace(SOURCE_PIN_PATH)
print({'source_mode': SOURCE_MODE, 'source_branch': GIT_REF, 'git_commit': resolved_commit, 'source_pin': str(SOURCE_PIN_PATH)})

In [ ]:
import importlib.metadata
import json
import sys

PIP_BASELINE_PATH = Path('/content/alqac-pip-check-baseline.json')

def pip_check_issues():
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'check'],
        text=True, capture_output=True, check=False,
    )
    output = '\n'.join(part.strip() for part in (result.stdout, result.stderr) if part.strip())
    issues = set(output.splitlines()) if result.returncode else set()
    return issues, output

if PIP_BASELINE_PATH.exists():
    pip_baseline = json.loads(PIP_BASELINE_PATH.read_text(encoding='utf-8'))
else:
    baseline_issues, _ = pip_check_issues()
    pip_baseline = {
        'issues': sorted(baseline_issues),
        'torch_version': importlib.metadata.version('torch'),
    }
    PIP_BASELINE_PATH.write_text(json.dumps(pip_baseline, indent=2), encoding='utf-8')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'])

torch_version_after = importlib.metadata.version('torch')
if torch_version_after != pip_baseline['torch_version']:
    raise RuntimeError(
        f"Colab torch changed from {pip_baseline['torch_version']} to {torch_version_after}; restart the runtime."
    )
after_issues, after_output = pip_check_issues()
new_issues = sorted(after_issues - set(pip_baseline['issues']))
if new_issues:
    raise RuntimeError('ALQAC dependency installation introduced new conflicts:\n- ' + '\n- '.join(new_issues))
if after_issues:
    print('WARNING: Colab image has pre-existing pip conflicts; no new conflicts were introduced.')
    print(after_output)
else:
    print('pip check: PASS (no broken requirements)')
print({'torch_preserved': torch_version_after, 'new_pip_conflicts': len(new_issues)})

## Restore reusable artifacts and validate the runtime

SQLite and model/index reads use local Colab storage. Verified copies are restored from Drive; live API writes are backed up to Drive after every network attempt.

In [ ]:
import json
import torch
from alqac2026.artifacts import DriveArtifactLayout, restore_directory, restore_sqlite_cache
from alqac2026.config import load_config
from alqac2026.data import load_law_corpus
from alqac2026.law_retrieval import law_index_fingerprint

if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before continuing.')
gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
ram_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024 ** 3)
local_free_gb = shutil.disk_usage('/content').free / (1024 ** 3)
if gpu_memory_gb < 14:
    raise RuntimeError(f'At least 14 GB GPU memory is required; got {gpu_memory_gb:.1f} GB')
if ram_gb < 10 or local_free_gb < 25:
    raise RuntimeError(f'Insufficient runtime storage: RAM={ram_gb:.1f} GB, local_free={local_free_gb:.1f} GB')
print({'gpu': gpu_name, 'gpu_memory_gb': round(gpu_memory_gb, 1), 'ram_gb': round(ram_gb, 1), 'local_free_gb': round(local_free_gb, 1), 'torch': torch.__version__})

CONFIG_PATH = PROJECT_ROOT / 'configs' / f'{EXPERIMENT}.yaml'
config = load_config(CONFIG_PATH)
for revision in (config['law_retrieval'].get('embedding_revision'), config['law_retrieval'].get('reranker_revision'), config['prediction'].get('revision')):
    if not isinstance(revision, str) or not re.fullmatch(r'[0-9a-fA-F]{40}', revision):
        raise ValueError(f'Model revision must be a pinned 40-character commit: {revision}')
articles = load_law_corpus(PROJECT_ROOT / config['paths']['corpus'])
index_fingerprint = law_index_fingerprint(config['law_retrieval'], articles)
layout = DriveArtifactLayout(DRIVE_ROOT, TRACK, RUN_ID)
LOCAL_RUNTIME = Path('/content/alqac-runtime')
LOCAL_CACHE = LOCAL_RUNTIME / 'cache' / 'case_api.sqlite'
LOCAL_INDEX = LOCAL_RUNTIME / 'law_index' / index_fingerprint
DRIVE_INDEX = DRIVE_ROOT / 'indexes' / 'law' / index_fingerprint
LOCAL_CACHE.parent.mkdir(parents=True, exist_ok=True)
LOCAL_INDEX.mkdir(parents=True, exist_ok=True)
cache_restored = restore_sqlite_cache(layout.cache_backup, LOCAL_CACHE)
index_restored = restore_directory(DRIVE_INDEX, LOCAL_INDEX)

free_gb = shutil.disk_usage(DRIVE_ROOT).free / (1024 ** 3)
PERSIST_MODEL_CACHE = free_gb >= MODEL_CACHE_FREE_GB
LOCAL_HF = LOCAL_RUNTIME / 'huggingface'
LOCAL_HF.mkdir(parents=True, exist_ok=True)
if PERSIST_MODEL_CACHE:
    restore_directory(DRIVE_ROOT / 'model_cache', LOCAL_HF)
os.environ['HF_HOME'] = str(LOCAL_HF)
hf_token = secret_or_none('HF_TOKEN')
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
else:
    os.environ.pop('HF_TOKEN', None)
hf_token = None
print({'cache_restored': cache_restored, 'index_restored': index_restored, 'index_fingerprint': index_fingerprint, 'persist_model_cache': PERSIST_MODEL_CACHE, 'drive_free_gb': round(free_gb, 1)})

In [ ]:
from alqac2026.config import sha256_file
from alqac2026.data import load_inference_cases

if TRACK == 'public':
    INPUT_PATH = PROJECT_ROOT / 'data/raw/ALQAC2026_public_test.json'
    SELECTION_PROFILE = None
else:
    INPUT_PATH = layout.private_input
    if not INPUT_PATH.is_file():
        raise FileNotFoundError(f'Place the official Private input at {INPUT_PATH}')
    private_payload = json.loads(INPUT_PATH.read_text(encoding='utf-8'))
    if len(private_payload) != 60 or any(set(item) != {'case_id', 'case_query'} for item in private_payload):
        raise ValueError('Private input must contain 60 objects with exactly case_id and case_query.')
    if sha256_file(INPUT_PATH) != PRIVATE_INPUT_SHA256:
        raise ValueError('Private input SHA-256 does not match the reviewed organizer file.')
    SELECTION_PROFILE = DRIVE_ROOT / 'runs/public' / PUBLIC_SELECTION_RUN_ID / 'selection_profile.json'
    if not SELECTION_PROFILE.is_file():
        raise FileNotFoundError(f'Missing Public selection profile: {SELECTION_PROFILE}')

all_cases = load_inference_cases(INPUT_PATH)
expected_cases = 50 if TRACK == 'public' else 60
assert len(all_cases) == expected_cases
print({'input': str(INPUT_PATH), 'cases': len(all_cases), 'selection_profile': str(SELECTION_PROFILE) if SELECTION_PROFILE else None})

In [ ]:
from alqac2026.case_retrieval import SQLiteEvidenceCache, build_api_plan

planned_cases = all_cases[:2] if RUN_MODE == 'smoke' else all_cases
cache = SQLiteEvidenceCache(LOCAL_CACHE)
cache.integrity_check()
api_plan = build_api_plan(planned_cases, cache, max_queries=2, approved_max_network_calls=APPROVED_MAX_NETWORK_CALLS)
cache.close()
print({key: api_plan[key] for key in ('logical_queries', 'cache_hits', 'cache_misses', 'known_local_cumulative_attempts')})

if TRACK == 'public':
    RUN_NETWORK_CAP = 0
elif RUN_MODE == 'smoke':
    RUN_NETWORK_CAP = 4
elif RUN_MODE == 'resume':
    if APPROVED_MAX_NETWORK_CALLS is None:
        raise ValueError('Set APPROVED_MAX_NETWORK_CALLS explicitly before resume.')
    RUN_NETWORK_CAP = APPROVED_MAX_NETWORK_CALLS
else:
    RUN_NETWORK_CAP = APPROVED_MAX_NETWORK_CALLS or (api_plan['cache_misses'] + RETRY_RESERVE)
print({'execution_mode': EXECUTION_MODE, 'approved_network_cap': RUN_NETWORK_CAP})

In [ ]:
from alqac2026.artifacts import backup_directory

RUNTIME_REPORT = layout.run_dir / 'runtime_check.json'
RUNTIME_REPORT.parent.mkdir(parents=True, exist_ok=True)
runtime_command = [
    sys.executable, 'scripts/check_runtime.py',
    '--config', str(CONFIG_PATH),
    '--input', str(PROJECT_ROOT / 'data/raw/ALQAC2026_public_test.json'),
    '--output', str(RUNTIME_REPORT),
    '--law-index-dir', str(LOCAL_INDEX),
]
if ADAPTER_PATH:
    runtime_command.extend(['--adapter-path', str(ADAPTER_PATH)])
subprocess.check_call(runtime_command, cwd=PROJECT_ROOT)
runtime_report = json.loads(RUNTIME_REPORT.read_text(encoding='utf-8'))
assert runtime_report['status'] == 'PASS'
assert runtime_report['api_network_attempts'] == 0
source_pin = json.loads(SOURCE_PIN_PATH.read_text(encoding='utf-8'))
if source_pin.get('commit') != resolved_commit:
    raise RuntimeError('Runtime check source does not match source_pin.json')
source_pin.update({'runtime_check_status': 'PASS', 'runtime_source_sha256': runtime_report['source_sha256'], 'runtime_config_sha256': runtime_report['config_sha256']})
source_pin_temp = SOURCE_PIN_PATH.with_suffix('.json.tmp')
source_pin_temp.write_text(json.dumps(source_pin, indent=2), encoding='utf-8')
source_pin_temp.replace(SOURCE_PIN_PATH)
backup_directory(LOCAL_INDEX, DRIVE_INDEX)
if PERSIST_MODEL_CACHE:
    backup_directory(LOCAL_HF, DRIVE_ROOT / 'model_cache')
print({'runtime_check': runtime_report['status'], 'api_network_attempts': 0})

## Execute and validate

`ALQAC_TEAM_TOKEN` is read and exported only inside this cell and only for `TRACK='private'`, `EXECUTION_MODE='live'`. The pinned source commit is shared by runtime-check, smoke, full, and resume; smoke and full still use different run directories.

In [ ]:
RUN_DIR = None
if RUN_MODE != 'runtime_check':
    effective_run_id = f'{RUN_ID}-smoke' if RUN_MODE == 'smoke' else RUN_ID
    RUN_DIR = DRIVE_ROOT / 'runs' / TRACK / effective_run_id
    if RUN_MODE in {'smoke', 'full'} and (RUN_DIR / 'manifest.json').exists():
        raise FileExistsError(f'Run already exists; use RUN_MODE=resume or a new RUN_ID: {RUN_DIR}')
    if RUN_MODE == 'resume' and not (RUN_DIR / 'manifest.json').exists():
        raise FileNotFoundError(f'No resumable run found: {RUN_DIR}')

    if EXECUTION_MODE == 'live':
        team_token = secret_or_none('ALQAC_TEAM_TOKEN')
        if not team_token:
            raise ValueError('Missing Colab secret: ALQAC_TEAM_TOKEN')
        os.environ['ALQAC_TEAM_TOKEN'] = team_token
    else:
        os.environ.pop('ALQAC_TEAM_TOKEN', None)

    runner_script = 'scripts/run_public.py' if TRACK == 'public' else 'scripts/run_private.py'
    command = [
        sys.executable, runner_script,
        '--config', str(CONFIG_PATH),
        '--input', str(INPUT_PATH),
        '--resume-run', str(RUN_DIR),
        '--execution-mode', EXECUTION_MODE,
        '--cache-db', str(LOCAL_CACHE),
        '--cache-backup-db', str(layout.cache_backup),
        '--law-index-dir', str(LOCAL_INDEX),
        '--max-network-calls', str(RUN_NETWORK_CAP),
    ]
    if RUN_MODE == 'smoke':
        command.extend(['--limit', '2'])
    if SELECTION_PROFILE is not None:
        command.extend(['--selection-profile', str(SELECTION_PROFILE)])
    if ADAPTER_PATH:
        command.extend(['--adapter-path', str(ADAPTER_PATH)])
    try:
        subprocess.check_call(command, cwd=PROJECT_ROOT)
    finally:
        team_token = None
        os.environ.pop('ALQAC_TEAM_TOKEN', None)

In [ ]:
from alqac2026.artifacts import export_run

if RUN_DIR is not None:
    manifest = json.loads((RUN_DIR / 'manifest.json').read_text(encoding='utf-8'))
    validation = json.loads((RUN_DIR / 'validation.json').read_text(encoding='utf-8'))
    api_stats = json.loads((RUN_DIR / 'api_stats.json').read_text(encoding='utf-8'))
    run_environment = json.loads((RUN_DIR / 'environment.json').read_text(encoding='utf-8'))
    expected = 2 if RUN_MODE == 'smoke' else expected_cases
    assert manifest['run']['status'] == 'completed'
    assert manifest['source_sha256'] == runtime_report['source_sha256']
    assert run_environment['cuda'] == runtime_report['environment']['cuda']
    assert manifest['run']['completed'] == expected
    assert validation['status'] == 'PASS' and validation['cases'] == expected
    if TRACK == 'public':
        assert api_stats['run_network_attempts'] == 0
    submission_path = RUN_DIR / 'submission.json'
    assert submission_path.stat().st_size <= 10 * 1024 * 1024
    print({'run_dir': str(RUN_DIR), 'cases': expected, 'validation': 'PASS', 'network_attempts': api_stats['run_network_attempts']})
    if RUN_MODE in {'full', 'resume'}:
        export_path = export_run(RUN_DIR, DRIVE_ROOT / 'exports' / RUN_ID)
        print({'export_dir': str(export_path), 'leaderboard_upload': 'manual only'})

## Recommended execution order

1. Public runtime gate: choose a new `RUN_ID`, then set `TRACK='public'`, `RUN_MODE='runtime_check'`, `EXECUTION_MODE='cache-only'`. This pins the current `TuanAnh` commit.
2. Public smoke: change only `RUN_MODE='smoke'`. Confirm two completed predictions and zero network attempts.
3. Public full: keep the same `RUN_ID` and set `RUN_MODE='full'`. The run produces 50 predictions, metrics, `selection_profile.json`, and an optional manually uploadable Public file.
4. Private runtime gate: set `TRACK='private'`, `EXECUTION_MODE='live'`, and a new Private `RUN_ID`. Keep `RUN_MODE='runtime_check'`; the notebook pins `TuanAnh` and does not read the organizer token.
5. Private smoke: set `RUN_MODE='smoke'`; the hard cap is four attempts and the limited output must never be submitted.
6. Private full: set `RUN_MODE='full'`. The notebook reuses smoke cache and approves current misses plus four retry attempts.
7. Upload only `exports/<RUN_ID>/submission.json`. Select **Private Test**, enter a never-used run name, click **Check format**, review the result, then confirm manually.